In [2]:
"""
Step 3: proper Granger causality test, replacing the naive lag-scan with
something that correctly handles the shared-trend problem.

Workflow:
  1. ADF unit-root test on each raw series, PLUS an explicit linear-trend
     test (does this series have a statistically significant slope over
     time). These are checking two different things -- ADF asks "does this
     look like a random walk", the trend test asks "is there a real slope
     over the whole period" -- and a series can pass the first while
     failing the second (e.g. day-to-day values aren't a random walk, but
     there's still a slow 8-year drift). Catching ONLY unit roots and
     missing a real deterministic trend is exactly what let the original
     version of this script wave through a shared-trend confound.
  2. If EITHER check flags a problem, difference BOTH series in a pair by
     the same order (needed for a valid joint model -- can't compare a
     once-differenced series against a twice-differenced one).
  3. Use VAR order selection (AIC and BIC) to pick a defensible lag, instead
     of scanning every lag from -84 to +84 and picking whichever looks best
     (that would just be a new form of the multiple-comparisons problem).
  4. Run the actual Granger causality test at the AIC-selected lag and the
     BIC-selected lag, in BOTH directions:
       - does CVE volume Granger-cause burnout volume?
       - does burnout volume Granger-cause CVE volume? (reverse-direction
         sanity check -- if this is ALSO significant, that's a sign of a
         shared confound rather than a clean causal story)

Reads the CSV from build_q1_timeseries.py. No raw file access here.
"""

import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR
import os
import warnings

warnings.filterwarnings("ignore")  # statsmodels VAR is chatty about lag-order edge cases

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/q1_analysis/"
DAILY_CSV = os.path.join(OUT_DIR, "q1_daily_series.csv")

BAT_COLS = ["n_bat_ge1", "n_bat_ge2"]
CVE_COLS = ["n_cve_run1", "n_cve_run2"]

MAX_LAG_FOR_ORDER_SELECTION = 84
MAX_DIFF_ORDER = 2
ALPHA = 0.05


def adf_pvalue(series):
    return adfuller(series.dropna(), autolag="AIC")[1]


def trend_pvalue_and_slope(series):
    """OLS regression of the series against a plain time index (0,1,2,...).
    Returns (p_value, slope) for the slope coefficient. A significant
    p-value means a real deterministic trend exists over the period, even
    if the ADF unit-root test alone doesn't flag it."""
    s = series.dropna().values
    t = np.arange(len(s))
    slope, intercept, r, p, se = scipy_stats.linregress(t, s)
    return p, slope


def find_common_diff_order(series_a, series_b, label_a, label_b, max_d=MAX_DIFF_ORDER):
    """Difference both series together, by the same number of steps, until
    BOTH pass the unit-root test AND show no significant linear trend (or
    until max_d is hit)."""
    a, b = series_a.copy(), series_b.copy()
    d = 0
    while d <= max_d:
        p_a_unit, p_b_unit = adf_pvalue(a), adf_pvalue(b)
        p_a_trend, slope_a = trend_pvalue_and_slope(a)
        p_b_trend, slope_b = trend_pvalue_and_slope(b)

        unit_a = "no unit root" if p_a_unit < ALPHA else "UNIT ROOT"
        unit_b = "no unit root" if p_b_unit < ALPHA else "UNIT ROOT"
        trend_a = f"TREND (slope={slope_a:.5f}, p={p_a_trend:.4g})" if p_a_trend < ALPHA else "no significant trend"
        trend_b = f"TREND (slope={slope_b:.5f}, p={p_b_trend:.4g})" if p_b_trend < ALPHA else "no significant trend"

        print(f"    d={d}: {label_a}: ADF p={p_a_unit:.4g} ({unit_a}); linear trend check: {trend_a}")
        print(f"           {label_b}: ADF p={p_b_unit:.4g} ({unit_b}); linear trend check: {trend_b}")

        a_ok = (p_a_unit < ALPHA) and (p_a_trend >= ALPHA)
        b_ok = (p_b_unit < ALPHA) and (p_b_trend >= ALPHA)

        if a_ok and b_ok:
            return a, b, d

        a = a.diff().dropna()
        b = b.diff().dropna()
        d += 1
    print(f"    WARNING: still non-stationary/trending after d={max_d} -- proceeding anyway, "
          f"interpret results cautiously")
    return a, b, d


def select_lag_orders(df_pair, maxlags):
    model = VAR(df_pair)
    sel = model.select_order(maxlags=maxlags)
    return sel


def granger_at_lag(effect_series, cause_series, lag, direction_label):
    """data columns must be [effect, cause] -- statsmodels tests whether
    column 1 (cause) Granger-causes column 0 (effect)."""
    df_pair = pd.DataFrame({"effect": effect_series.values, "cause": cause_series.values})
    results = grangercausalitytests(df_pair, maxlag=[lag], verbose=False)
    fstat, pvalue, df_denom, df_num = results[lag][0]["ssr_ftest"]
    sig = "SIGNIFICANT" if pvalue < ALPHA else "not significant"
    print(f"    {direction_label} at lag={lag}: F={fstat:.4f}, p={pvalue:.4g}  -> {sig} (alpha={ALPHA})")
    return fstat, pvalue


def main():
    if not os.path.exists(DAILY_CSV):
        print(f"Could not find {DAILY_CSV}. Run build_q1_timeseries.py first.")
        return

    daily = pd.read_csv(DAILY_CSV)
    print(f"Loaded {len(daily)} days from {DAILY_CSV}")

    summary_rows = []

    for bat_col in BAT_COLS:
        for cve_col in CVE_COLS:
            print(f"\n{'=' * 70}")
            print(f"{bat_col}  vs  {cve_col}")
            print("=" * 70)

            print("\n  Stationarity check + common differencing order:")
            bat_s, cve_s, d = find_common_diff_order(daily[bat_col], daily[cve_col], bat_col, cve_col)
            print(f"  Using d={d} for both series in this pair")

            df_pair = pd.DataFrame({bat_col: bat_s.values, cve_col: cve_s.values})

            print(f"\n  Selecting lag order (AIC/BIC, up to {MAX_LAG_FOR_ORDER_SELECTION} days):")
            sel = select_lag_orders(df_pair, MAX_LAG_FOR_ORDER_SELECTION)
            aic_lag = max(int(sel.aic), 1)
            bic_lag = max(int(sel.bic), 1)
            print(f"  AIC-selected lag: {aic_lag}   |   BIC-selected lag: {bic_lag}")

            print(f"\n  Granger causality -- does CVE lead burnout?")
            for lag, crit_name in [(aic_lag, "AIC"), (bic_lag, "BIC")]:
                fstat, pvalue = granger_at_lag(bat_s, cve_s, lag, f"CVE -> burnout ({crit_name} lag)")
                summary_rows.append({
                    "bat_threshold": bat_col, "cve_threshold": cve_col,
                    "diff_order": d, "criterion": crit_name, "lag": lag,
                    "direction": "cve_to_bat", "f_stat": round(fstat, 4), "p_value": round(pvalue, 6),
                })

            print(f"\n  Reverse check -- does burnout lead CVE? (should NOT be significant for a clean story)")
            for lag, crit_name in [(aic_lag, "AIC"), (bic_lag, "BIC")]:
                fstat, pvalue = granger_at_lag(cve_s, bat_s, lag, f"burnout -> CVE ({crit_name} lag)")
                summary_rows.append({
                    "bat_threshold": bat_col, "cve_threshold": cve_col,
                    "diff_order": d, "criterion": crit_name, "lag": lag,
                    "direction": "bat_to_cve", "f_stat": round(fstat, 4), "p_value": round(pvalue, 6),
                })

    summary_df = pd.DataFrame(summary_rows)
    out_path = os.path.join(OUT_DIR, "q1_granger_causality_results_v2.csv")
    summary_df.to_csv(out_path, index=False)

    print(f"\n{'=' * 70}")
    print("FULL SUMMARY")
    print("=" * 70)
    print(summary_df.to_string(index=False))
    print(f"\nSaved -> {out_path}")

    print(f"\n{'=' * 70}")
    print("HOW TO READ THIS")
    print("=" * 70)
    print("Look for rows where direction='cve_to_bat' is significant (p < 0.05) AND")
    print("the matching direction='bat_to_cve' row for the same combination/criterion")
    print("is NOT significant. That's a clean, directional result: CVE volume helps")
    print("predict burnout volume beyond what burnout's own past already tells you,")
    print("and it doesn't work the other way around.")
    print("\nIf BOTH directions are significant, that points to a shared confound (like")
    print("the trend issue from the raw lag scan) rather than a clean causal story --")
    print("though differencing above should have already removed most of that.")
    print("\nIf NEITHER direction is significant at any threshold, that's a valid, honest")
    print("null result: once trend is removed, CVE volume doesn't have detectable")
    print("predictive power over burnout volume at the whole-corpus level.")


if __name__ == "__main__":
    main()

Loaded 3071 days from /Users/nadia/Desktop/redditRun_june/q1_analysis/q1_daily_series.csv

n_bat_ge1  vs  n_cve_run1

  Stationarity check + common differencing order:
    d=0: n_bat_ge1: ADF p=0.01121 (no unit root); linear trend check: TREND (slope=0.00200, p=1.343e-89)
           n_cve_run1: ADF p=1.031e-06 (no unit root); linear trend check: TREND (slope=-0.00128, p=3.271e-14)
    d=1: n_bat_ge1: ADF p=3.68e-28 (no unit root); linear trend check: no significant trend
           n_cve_run1: ADF p=2.435e-30 (no unit root); linear trend check: no significant trend
  Using d=1 for both series in this pair

  Selecting lag order (AIC/BIC, up to 84 days):
  AIC-selected lag: 55   |   BIC-selected lag: 20

  Granger causality -- does CVE lead burnout?
    CVE -> burnout (AIC lag) at lag=55: F=1.7844, p=0.0003732  -> SIGNIFICANT (alpha=0.05)
    CVE -> burnout (BIC lag) at lag=20: F=3.2063, p=1.917e-06  -> SIGNIFICANT (alpha=0.05)

  Reverse check -- does burnout lead CVE? (should NOT be s